# Download Integration

This notebook demonstrates the **Download** stage of the Foreign Whispers dubbing pipeline.
It downloads a YouTube video and its closed captions via `yt-dlp` through the FastAPI backend.

**Prerequisites:**
- The Docker stack must be running (`docker compose --profile nvidia up -d`).
- The API should be accessible at `http://localhost:8080`.

## Setup

In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

IMAGES_DIR = Path("images")
IMAGES_DIR.mkdir(exist_ok=True)

# Load .env (LOGFIRE_TOKEN, HF_TOKEN, etc.)
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

# Optional: Logfire tracing (no-op shim if unavailable)
try:
    import logfire
    logfire.configure(service_name="foreign-whispers-download")
    print("Logfire tracing enabled.")
except Exception:
    class _NoopSpan:
        def __enter__(self): return self
        def __exit__(self, *a): pass
    class _noop:
        @staticmethod
        def span(name, **kw): return _NoopSpan()
        @staticmethod
        def info(*a, **kw): pass
    logfire = _noop()
    print("Logfire not configured — using no-op shim.")

Project root: /home/william-zheng/Documents/Programming/Python/AI_course/foreign-whispers
Logfire not configured — using no-op shim.


In [2]:
from foreign_whispers.client import FWClient

fw = FWClient("http://localhost:8080")
fw.healthz()

{'status': 'ok'}

## Download Video and Captions

The API wraps `yt-dlp` to download the video MP4 and extract any available closed captions.
The `fw.download()` call returns a dict with `video_id`, `title`, and `caption_segments`.

In [3]:
VIDEO_URL = "https://www.youtube.com/watch?v=GYQ5yGV_-Oc"

with logfire.span("download", video_url=VIDEO_URL):
    dl = fw.download(VIDEO_URL)

print(f"Video ID:       {dl['video_id']}")
print(f"Title:          {dl['title']}")
print(f"Caption count:  {len(dl['caption_segments'])}")
print()
print("First 5 caption segments:")
for seg in dl["caption_segments"][:5]:
    dur = seg.get("duration", 0)
    print(f"  [{seg['start']:.2f}s, {dur:.2f}s] {seg['text']}")

HTTPError: 500 Server Error: Internal Server Error for url: http://localhost:8080/api/download

## Inspect Downloaded Artifacts

The download stage writes files into the `pipeline_data/api/` directory tree:

- `pipeline_data/api/videos/` — source MP4 files
- `pipeline_data/api/youtube_captions/` — extracted caption JSON files

In [ ]:
videos_dir = PROJECT_ROOT / "pipeline_data" / "api" / "videos"
captions_dir = PROJECT_ROOT / "pipeline_data" / "api" / "youtube_captions"

print("Videos:")
for f in sorted(videos_dir.iterdir()):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name}  ({size_mb:.1f} MB)")

print()
print("YouTube captions:")
for f in sorted(captions_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}  ({size_kb:.1f} KB)")

## Visualize Caption Timeline

Plot the downloaded caption segments as horizontal bars on a timeline to visualize
their temporal distribution across the video.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 3))
for seg in dl["caption_segments"]:
    dur = seg.get("duration", 0)
    if dur > 0:
        ax.barh(0, dur, left=seg["start"], height=0.5, color="steelblue", alpha=0.7)
ax.set_xlabel("Time (s)")
ax.set_title("YouTube Caption Timeline")
ax.set_yticks([])
fig.tight_layout()
fig.savefig(str(IMAGES_DIR / "caption_timeline.png"), dpi=150)
plt.show()

## Summary

The download stage produced:

- **MP4 video file** in `pipeline_data/api/videos/`
- **YouTube captions JSON** in `pipeline_data/api/youtube_captions/`

These artifacts are consumed by the next pipeline stages (transcription, translation, TTS, and stitching).